# Fine Tuning SAM Audio

In [1]:
from anatolian_sam.model_utils import load_sam_audio
from anatolian_sam.latents_dataloader import (
    TurkishMusicLatentsDataset,
    pad_collate_fn,
)
from anatolian_sam.flow_utils import expand_to_256

/teamspace/studios/this_studio/anatolian-SAM/.venv/lib/python3.11/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/teamspace/studios/this_studio/anatolian-SAM/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
WARNING[XFORMERS]: xFormers can't load C++/CUDA extensions. xFormers was built for:
    PyTorch 2.10.0+cu128 with CUDA 1208 (you have 2.11.0+cu130)
    Python  3.10.19 (you have 3.11.15)
  Please reinstall xformers (see https://github.com/facebookresearch/xformers#installing-xformers)
  Memory-efficient attention, SwiGLU, sparse and more won't be available

## Load SAM Audio and Processor

In [2]:
base_model, processor = load_sam_audio()

Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 22696.45it/s]
/teamspace/studios/this_studio/anatolian-SAM/.venv/lib/python3.11/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
/teamspace/studios/this_studio/anatolian-SAM/.venv/lib/python3.11/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/teamspace/studios/this_studio/anatolian-SAM/.venv/lib/python3.11/site-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr

## Setup PEFT/LoRA Fine-Tune

In [3]:
import torch

base_model.text_encoder.requires_grad_(False)

with torch.no_grad():
    text_features, text_mask = base_model.text_encoder(["zurna"])

print("Text embeddings shape:", text_features.shape)

Text embeddings shape: torch.Size([1, 4, 768])


### Custom Dataset

In [4]:
train_dataset = TurkishMusicLatentsDataset(
    jsonl_path="../data/train_metadata_tr.jsonl",
    data_base_path="../data/latents",
)

test_dataset = TurkishMusicLatentsDataset(
    jsonl_path="../data/val_metadata_tr.jsonl",
    data_base_path="../data/latents",
)

Loaded 576 audio tuples from train_metadata_tr.jsonl
Loaded 144 audio tuples from val_metadata_tr.jsonl


In [5]:
BATCH_SIZE = 8
ALPHA = 16
RANK = 16
LEARNING_RATE = 5e-5
EPOCHS = 100
DROPOUT = 0.05

In [6]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    collate_fn=pad_collate_fn,
    pin_memory=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    collate_fn=pad_collate_fn,
    pin_memory=True,
)

In [7]:
for batch in train_loader:
    print("Batch keys:", batch.keys())
    print("Mixture latent shape:", batch["mixture_latent"].shape)
    print("Target latent shape:", batch["target_latent"].shape)
    print("Prompts:", batch["prompt"])
    break  # Just check the first batch

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

Batch keys: dict_keys(['mixture_latent', 'target_latent', 'target_filename', 'prompt'])
Mixture latent shape: torch.Size([8, 128, 125])
Target latent shape: torch.Size([8, 128, 125])
Prompts: ['bağlama', 'bağlama', 'bağlama', 'bağlama', 'zurna', 'zurna', 'bağlama', 'zurna']


### Wrap with PEFT/LoRA

In [8]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=RANK,
    lora_alpha=ALPHA,
    # The developers of SAMAudio named the attention layers in their DiT (the DiT handles the actual latent generation) wq and wv
    target_modules=["wq", "wv"],
    lora_dropout=DROPOUT,
    bias="none",
)

peft_model = get_peft_model(base_model, lora_config)
peft_model.print_trainable_parameters()

trainable params: 4,194,304 || all params: 6,468,359,530 || trainable%: 0.0648


### Initialize Flow Matching Scheduler and Accelerate

In [9]:
from diffusers import FlowMatchEulerDiscreteScheduler
from diffusers.optimization import get_cosine_schedule_with_warmup
from accelerate import Accelerator
from torch.optim import AdamW
import math

# initialize flow-matching scheduler
scheduler = FlowMatchEulerDiscreteScheduler(
    num_train_timesteps=1000,
    shift=1.0,  # standard shift value for flow matching
)

# initialize accelerator for vram and device management
accelerator = Accelerator(gradient_accumulation_steps=4, mixed_precision="bf16")

optimizer = AdamW(peft_model.parameters(), lr=LEARNING_RATE)

# CRITICAL: Because we are accumulating gradients over 4 batches,
# the optimizer only takes a "step" once every 4 forward passes.
num_update_steps_per_epoch = math.ceil(
    len(train_loader) / accelerator.gradient_accumulation_steps
)
max_train_steps = EPOCHS * num_update_steps_per_epoch

# 2. Define Warmup Steps
# A standard rule of thumb for LoRA is warming up for 5% to 10% of total training steps.
num_warmup_steps = int(max_train_steps * 0.10)

# 3. Initialize the Scheduler
lr_scheduler = get_cosine_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=max_train_steps,
)

# Pass everything to accelerator.prepare
peft_model, optimizer, train_loader, test_loader, lr_scheduler = accelerator.prepare(
    peft_model, optimizer, train_loader, test_loader, lr_scheduler
)

print("Flow-Matching Scheduler and Accelerator Initialized!")
print(f"Total Training Steps: {max_train_steps}")
print(f"Warmup Steps: {num_warmup_steps}")

Flow-Matching Scheduler and Accelerator Initialized!
Total Training Steps: 1800
Warmup Steps: 180


## Custom LoRA Flow-Matching Training Loop

In [10]:
import inspect

# Inspect the forward method of the underlying base model
signature = inspect.signature(base_model.forward)

print("SAM Audio Forward Signature:")
for param in signature.parameters.values():
    print(f"- {param.name}: {param.default}")

SAM Audio Forward Signature:
- noisy_audio: <class 'inspect._empty'>
- audio_features: <class 'inspect._empty'>
- text_features: <class 'inspect._empty'>
- time: <class 'inspect._empty'>
- masked_video_features: None
- text_mask: None
- anchor_ids: None
- anchor_alignment: None
- audio_pad_mask: None


In [11]:
import wandb
import os

wandb.init(
    entity="zeerafle-sivas-cumhuriyet-university",
    project="turkish-sam-audio",
    name="two-instruments-lora-flow-matching-run",
    config={
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "lora_rank": RANK,
        "lora_alpha": ALPHA,
        "lora_dropout": DROPOUT,
    },
    dir="../wandb",
)

# Setup Checkpointing Directory
save_directory = "../checkpoints/turkish_sam_audio_lora_best"
os.makedirs(save_directory, exist_ok=True)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: zeerafle (zeerafle-sivas-cumhuriyet-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [12]:
import torch.nn.functional as F
from tqdm.autonotebook import tqdm


# --- EARLY STOPPING PARAMETERS ---
patience = 7
min_delta = 0.0001  # Minimum change required to count as an improvement
patience_counter = 0  # Tracks epochs without improvement
best_val_loss = float("inf")
global_step = 0


for epoch in range(EPOCHS):
    # 1. TRAINING PHASE
    peft_model.train()
    epoch_loss = 0.0  # Accumulator to calculate average loss per epoch

    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS - 1}")

    # train_dataloader yields the Tuple: Mixture, Target, Text Embeddings
    for step, batch in enumerate(pbar):
        # encode text
        with torch.no_grad():
            text_features, text_mask = base_model.text_encoder(batch["prompt"])
            text_features = text_features.to(accelerator.device)
            text_mask = text_mask.to(accelerator.device)

        # accelerate.accumulate handles the 16-step gradient hold for your T4
        with accelerator.accumulate(peft_model):
            # 1. Unpack the Pre-Computed Tensors
            # Conditioner 1: [Batch, 125, 128]
            mixture_latent = batch["mixture_latent"].transpose(1, 2)
            # Ground Truth x_1: [Batch, 125, 128]
            target_latent = batch["target_latent"].transpose(1, 2)

            # Time Sampling
            # You randomly select a continuous timestep (t) between 0 and 1[cite: 1062].
            t = torch.rand((BATCH_SIZE,), device=accelerator.device)
            t_expanded = t.view(BATCH_SIZE, 1, 1)  # Reshape for tensor broadcasting

            # additional step, Meta's trick (https://github.com/facebookresearch/sam-audio/blob/68b48d48fff1ad776d3afefbe634eb5f5d60ba7b/sam_audio/model/model.py#L184)
            mixture_256 = expand_to_256(mixture_latent)
            target_256 = expand_to_256(target_latent)

            # generate Noise and Noisy State in the 256-dimensional space
            noise_256 = torch.randn_like(target_256)
            x_t_256 = (1 - t_expanded) * noise_256 + t_expanded * target_256

            # The Forward Pass
            # Wrap the forward pass to automatically handle mixed precision casting
            with accelerator.autocast():
                # You feed the noisy state (x_t), the timestep (t), and your two completely clean conditioners (the Mixture Latent and the Text Prompt) into your peft_model[cite: 1064].
                predicted_velocity = peft_model(
                    noisy_audio=x_t_256,
                    time=t,
                    audio_features=mixture_256,
                    text_features=text_features,
                    text_mask=text_mask,
                )

                # calculate the True Vector Field
                # Flow-matching models predict the trajectory pointing from noise to target.
                true_velocity = target_256 - noise_256
                # Vector Field Discrepancy Loss
                # Calculate the MSE between the predicted trajectory and the true trajectory[cite: 1107].
                loss = F.mse_loss(predicted_velocity, true_velocity)

            # the Update
            # Push the gradients back strictly to the tiny q_proj and v_proj LoRA weights[cite: 1066].
            accelerator.backward(loss)

            # Optimizer automatically respects the gradient_accumulation_steps=16 we set earlier
            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad()

        # Update the tqdm progress bar with the current loss
        pbar.set_postfix(loss=f"{loss.item():.4f}")

        # Log Step-Level Metrics to W&B
        wandb.log(
            {
                "train/step_loss": loss.item(),
                "train/learning_rate": lr_scheduler.get_last_lr()[0],
                "global_step": global_step,
            }
        )

        epoch_loss += loss.item()
        global_step += 1

    avg_train_loss = epoch_loss / len(train_loader)

    # 2. VALIDATION PHASE
    peft_model.eval()  # Switch to evaluation mode (locks dropout, batchnorm, etc.)
    val_loss = 0.0

    val_pbar = tqdm(test_loader, desc=f"Val Epoch {epoch}/{EPOCHS - 1}", colour="green")

    for step, batch in enumerate(val_pbar):
        # We use torch.no_grad() for the entire validation block to save memory
        with torch.no_grad():
            text_features, text_mask = base_model.text_encoder(batch["prompt"])
            text_features = text_features.to(accelerator.device)
            text_mask = text_mask.to(accelerator.device)

            mixture_latent = batch["mixture_latent"].transpose(1, 2)
            target_latent = batch["target_latent"].transpose(1, 2)

            t = torch.rand((BATCH_SIZE,), device=accelerator.device)
            t_expanded = t.view(BATCH_SIZE, 1, 1)

            mixture_256 = expand_to_256(mixture_latent)
            target_256 = expand_to_256(target_latent)

            noise_256 = torch.randn_like(target_256)
            x_t_256 = (1 - t_expanded) * noise_256 + t_expanded * target_256

            with accelerator.autocast():
                predicted_velocity = peft_model(
                    noisy_audio=x_t_256,
                    time=t,
                    audio_features=mixture_256,
                    text_features=text_features,
                    text_mask=text_mask,
                )

                true_velocity = target_256 - noise_256
                v_loss = F.mse_loss(predicted_velocity, true_velocity)

            val_loss += v_loss.item()
            val_pbar.set_postfix(loss=f"{v_loss.item():.4f}")

    avg_val_loss = val_loss / len(test_loader)

    # 3. LOGGING & EARLY STOPPING LOGIC
    wandb.log(
        {
            "train/epoch_loss": avg_train_loss,
            "val/epoch_loss": avg_val_loss,
            "epoch": epoch,
        }
    )

    print(
        f"--- Epoch {epoch} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} ---"
    )

    # Early Stopping Check
    if avg_val_loss < (best_val_loss - min_delta):
        print(
            f"✅ Validation loss improved from {best_val_loss:.4f} to {avg_val_loss:.4f}. Saving checkpoint..."
        )
        best_val_loss = avg_val_loss
        patience_counter = 0  # Reset patience

        # Ensure multi-GPU operations are synced before saving
        accelerator.wait_for_everyone()

        if accelerator.is_main_process:
            unwrapped_model = accelerator.unwrap_model(peft_model)
            unwrapped_model.save_pretrained(save_directory, safe_serialization=True)

    else:
        patience_counter += 1
        print(
            f"⚠️ Validation loss did not improve. Patience: {patience_counter}/{patience}"
        )

        if patience_counter >= patience:
            print(
                "🛑 Early stopping triggered! Training halted to prevent overfitting."
            )
            break  # Break out of the epoch loop entirely

# Finish the W&B run cleanly
wandb.finish()

Epoch 0/99:   0%|          | 0/72 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Val Epoch 0/99:   0%|       

--- Epoch 0 | Train Loss: 0.6641 | Val Loss: 0.5496 ---
✅ Validation loss improved from inf to 0.5496. Saving checkpoint...


Epoch 1/99:   0%|          | 0/72 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARAL

--- Epoch 1 | Train Loss: 0.6333 | Val Loss: 0.5792 ---
⚠️ Validation loss did not improve. Patience: 1/7


...

Epoch 44/99:   0%|          | 0/72 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARA

--- Epoch 44 | Train Loss: 0.2990 | Val Loss: 0.2624 ---
⚠️ Validation loss did not improve. Patience: 6/7


Epoch 45/99:   0%|          | 0/72 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARA

--- Epoch 45 | Train Loss: 0.3004 | Val Loss: 0.2880 ---
⚠️ Validation loss did not improve. Patience: 7/7
🛑 Early stopping triggered! Training halted to prevent overfitting.


epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇████
global_step,▁▁▁▁▁▂▂▂▂▂▂▂▂▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇▇███
train/epoch_loss,█▇▇▇▆▅▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▂▁▂▁▁▁▁▁▁▁
train/learning_rate,▁▂▃▃▃▆▆▇▇█████████████████▇▇▇▇▇▇▇▇▇▆▆▆▆▆
train/step_loss,▆▆▇▅▆█▄▄▃▅▂▆▄▄▃▅▄▃▂▁▄▂▂▃▄▃▂▃▁▂▂▃▂▂▁▅▁▂▂▄
val/epoch_loss,▇█▇▇▅▅▄▄▄▄▃▃▃▃▃▃▃▂▂▃▁▂▃▂▂▁▁▂▁▂▁▁▁▁▁▂▁▂▁▁
epoch,45
global_step,3311
train/epoch_loss,0.30039
train/learning_rate,3e-05
train/step_loss,0.22849


In [13]:
# 1. Ensure # 1. Ensure all GPUs have finished their final calculations
accelerator.wait_for_everyone()

# 2. Define a DISTINCT output directory for the final epoch
# Notice the "_final" suffix to prevent overwriting your "_best" checkpoint
save_directory = "../checkpoints/turkish_sam_audio_lora_final"
os.makedirs(save_directory, exist_ok=True)

# 3. Unwrap and Save the final epoch state
if accelerator.is_main_process:
    unwrapped_model = accelerator.unwrap_model(peft_model)
    unwrapped_model.save_pretrained(
        save_directory,
        safe_serialization=True,
    )

print(f"Final epoch LoRA adapters successfully saved to: {save_directory}")

Final epoch LoRA adapters successfully saved to: ../checkpoints/turkish_sam_audio_lora_final
